# Demand Prediction Pipeline
**Goal**: Maximize R² score using ensemble of gradient boosting models

**Strategy**:
1. Feature engineering (timestamp, geohash, interactions)
2. Handle missing values natively (GBDT models)
3. Handle high-cardinality categoricals via label + target encoding
4. Ensemble: LightGBM + XGBoost + CatBoost with optimal blending

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

ModuleNotFoundError: No module named 'lightgbm'

## 1. Load Data

In [ ]:
train = pd.read_csv('./dataset/train.csv')
test = pd.read_csv('./dataset/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nMissing values in train:')
print(train.isnull().sum()[train.isnull().sum() > 0])
train.head()

## 2. Feature Engineering

In [ ]:
target = 'demand'
y_train = train[target].values
train_idx = train['Index'].values
test_idx = test['Index'].values

def engineer_features(df):
    out = df.copy()
    
    # --- Timestamp features ---
    parts = out['timestamp'].str.split(':', expand=True).astype(int)
    out['hour'] = parts[0]
    out['minute'] = parts[1]
    out['time_minutes'] = out['hour'] * 60 + out['minute']
    
    # Cyclical encoding (23:45 is close to 0:00)
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['min_sin'] = np.sin(2 * np.pi * out['time_minutes'] / 1440)
    out['min_cos'] = np.cos(2 * np.pi * out['time_minutes'] / 1440)
    
    # Time-of-day buckets
    out['time_bucket'] = pd.cut(out['hour'], bins=[-1,5,9,12,17,21,24],
                                labels=[0,1,2,3,4,5]).astype(int)
    
    # --- Geohash prefixes ---
    out['geo_prefix4'] = out['geohash'].str[:4]
    out['geo_prefix5'] = out['geohash'].str[:5]
    
    # --- Binary categoricals ---
    out['LargeVehicles_enc'] = (out['LargeVehicles'] == 'Allowed').astype(int)
    out['Landmarks_enc'] = (out['Landmarks'] == 'Yes').astype(int)
    
    # --- RoadType & Weather: label encode (NaN stays NaN for GBDT) ---
    out['RoadType_enc'] = out['RoadType'].map({'Residential': 0, 'Street': 1, 'Highway': 2})
    out['Weather_enc'] = out['Weather'].map({'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3})
    
    # --- Missing indicator ---
    out['Temperature_missing'] = out['Temperature'].isnull().astype(int)
    
    # --- Interaction features ---
    out['lanes_x_road'] = out['NumberofLanes'] * out['RoadType_enc']
    out['temp_x_weather'] = out['Temperature'] * out['Weather_enc']
    out['lanes_x_landmarks'] = out['NumberofLanes'] * out['Landmarks_enc']
    out['lanes_x_largeveh'] = out['NumberofLanes'] * out['LargeVehicles_enc']
    
    return out

train_fe = engineer_features(train)
test_fe = engineer_features(test)
print('Feature engineering done!')

In [ ]:
# --- Geohash label encoding (fit on train+test combined) ---
for col, prefix_len in [('geohash', None), ('geo_prefix4', None), ('geo_prefix5', None)]:
    all_vals = pd.concat([train_fe[col], test_fe[col]])
    le = LabelEncoder().fit(all_vals)
    train_fe[col + '_enc'] = le.transform(train_fe[col])
    test_fe[col + '_enc'] = le.transform(test_fe[col])

# --- Target encoding for geohash ---
geo_target_mean = train_fe.groupby('geohash')[target].mean()
geo_target_std = train_fe.groupby('geohash')[target].std().fillna(0)
geo_target_count = train_fe.groupby('geohash')[target].count()

global_mean = y_train.mean()

train_fe['geo_target_mean'] = train_fe['geohash'].map(geo_target_mean)
train_fe['geo_target_std'] = train_fe['geohash'].map(geo_target_std)
train_fe['geo_target_count'] = train_fe['geohash'].map(geo_target_count)

test_fe['geo_target_mean'] = test_fe['geohash'].map(geo_target_mean).fillna(global_mean)
test_fe['geo_target_std'] = test_fe['geohash'].map(geo_target_std).fillna(0)
test_fe['geo_target_count'] = test_fe['geohash'].map(geo_target_count).fillna(0)

print('Encoding done!')

In [ ]:
# --- Final feature list ---
features = [
    'day', 'hour', 'minute', 'time_minutes',
    'hour_sin', 'hour_cos', 'min_sin', 'min_cos', 'time_bucket',
    'geohash_enc', 'geo_prefix4_enc', 'geo_prefix5_enc',
    'geo_target_mean', 'geo_target_std', 'geo_target_count',
    'RoadType_enc', 'NumberofLanes', 'LargeVehicles_enc', 'Landmarks_enc',
    'Temperature', 'Temperature_missing', 'Weather_enc',
    'lanes_x_road', 'temp_x_weather', 'lanes_x_landmarks', 'lanes_x_largeveh',
]

X_train = train_fe[features].values
X_test = test_fe[features].values

print(f'Total features: {len(features)}')
print(features)

## 3. Model Training (5-Fold CV)

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

### 3a. LightGBM

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42,
}

oof_lgb = np.zeros(len(X_train))
pred_lgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(lgb_params, dtrain, num_boost_round=3000,
                      valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    oof_lgb[val_idx] = model.predict(X_val)
    pred_lgb += model.predict(X_test) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_lgb[val_idx]):.6f}')

print(f'\nLightGBM OOF R2: {r2_score(y_train, oof_lgb):.6f}')

### 3b. XGBoost

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 8,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'random_state': 42,
}

oof_xgb = np.zeros(len(X_train))
pred_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=features)
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
    dtest = xgb.DMatrix(X_test, feature_names=features)
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=3000,
                      evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=0)
    
    oof_xgb[val_idx] = model.predict(dval)
    pred_xgb += model.predict(dtest) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_xgb[val_idx]):.6f}')

print(f'\nXGBoost OOF R2: {r2_score(y_train, oof_xgb):.6f}')

### 3c. CatBoost

In [ ]:
oof_cat = np.zeros(len(X_train))
pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    model = CatBoostRegressor(
        iterations=3000, learning_rate=0.03, depth=8,
        l2_leaf_reg=3, random_seed=42, verbose=0,
        early_stopping_rounds=100, eval_metric='RMSE',
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
    
    oof_cat[val_idx] = model.predict(X_val)
    pred_cat += model.predict(X_test) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_cat[val_idx]):.6f}')

print(f'\nCatBoost OOF R2: {r2_score(y_train, oof_cat):.6f}')

## 4. Optimal Ensemble Blending

In [ ]:
best_r2 = -999
best_w = (0, 0, 0)

for w1 in np.arange(0.1, 0.9, 0.05):
    for w2 in np.arange(0.1, 0.9 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        r2 = r2_score(y_train, blend)
        if r2 > best_r2:
            best_r2 = r2
            best_w = (w1, w2, w3)

print(f'Optimal Weights -> LGB: {best_w[0]:.2f}, XGB: {best_w[1]:.2f}, CAT: {best_w[2]:.2f}')
print(f'\nIndividual R2 scores:')
print(f'  LightGBM:  {r2_score(y_train, oof_lgb):.6f}')
print(f'  XGBoost:   {r2_score(y_train, oof_xgb):.6f}')
print(f'  CatBoost:  {r2_score(y_train, oof_cat):.6f}')
print(f'  Ensemble:  {best_r2:.6f}')

## 5. Save Predictions

In [ ]:
final_pred = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand.csv', index=False)

print(f'Saved: predicted_demand.csv ({submission.shape[0]} rows)')
submission.head(10)

## 6. Save Approach Document

In [ ]:
approach_text = f"""================================================================================
                    DEMAND PREDICTION - APPROACH DOCUMENT
================================================================================

1. PROBLEM STATEMENT
--------------------
Predict the 'demand' (continuous float) for various geographic locations (geohash)
at specific timestamps, given road characteristics, weather, and temperature data.
This is a regression task, optimized for maximum R2 score.

Dataset: 77,299 train rows / 41,778 test rows / 11 columns


2. DATA OVERVIEW
----------------
Columns:
  - Index          : Row identifier (int)
  - geohash        : Geographic hash code (1,249 unique - HIGH cardinality)
  - day            : Day number (48 or 49)
  - timestamp      : Time in "H:M" format (96 unique, 15-min intervals)
  - demand         : TARGET variable (continuous float)
  - RoadType       : Categorical - Residential/Street/Highway (600 missing)
  - NumberofLanes  : Integer (1-5)
  - LargeVehicles  : Binary - Allowed/Not Allowed
  - Landmarks      : Binary - Yes/No
  - Temperature    : Float, degrees (2,495 missing)
  - Weather        : Categorical - Sunny/Rainy/Foggy/Snowy (797 missing)


3. FEATURE ENGINEERING
----------------------
a) Timestamp Decomposition:
   - Extracted 'hour' and 'minute' from "H:M" format
   - Created 'time_minutes' = hour*60 + minute
   - Cyclical sin/cos encoding for hour and minutes
   - Time bucket: binned hours into 6 periods

b) Geohash Engineering (High Cardinality - 1249 unique):
   - Label encoded the full geohash for tree models
   - Extracted geohash prefixes at 4-char and 5-char levels
   - Target encoding: mean, std, and count of demand per geohash

c) Categorical Encoding:
   - RoadType: mapped to 0/1/2 (NaN left as-is for GBDT)
   - Weather: mapped to 0/1/2/3 (NaN left as-is for GBDT)
   - LargeVehicles & Landmarks: binary encoded

d) Missing Value Strategy:
   - GBDT models handle NaN natively with optimal split direction
   - Added 'Temperature_missing' binary indicator

e) Interaction Features:
   - lanes_x_road, temp_x_weather, lanes_x_landmarks, lanes_x_largeveh

Total features: {len(features)}


4. MODELING APPROACH
--------------------
3-model ensemble with 5-Fold Cross-Validation:

a) LightGBM: lr=0.03, num_leaves=127, early_stop=100
   OOF R2: {r2_score(y_train, oof_lgb):.6f}

b) XGBoost: lr=0.03, max_depth=8, hist tree method
   OOF R2: {r2_score(y_train, oof_xgb):.6f}

c) CatBoost: lr=0.03, depth=8, l2_leaf_reg=3
   OOF R2: {r2_score(y_train, oof_cat):.6f}

d) Ensemble (Weighted Average):
   Weights -> LGB: {best_w[0]:.2f}, XGB: {best_w[1]:.2f}, CAT: {best_w[2]:.2f}
   Final OOF R2: {best_r2:.6f}


5. WHY THIS APPROACH MAXIMIZES R2
----------------------------------
- GBDT handles missing values natively (no imputation bias)
- Label encoding works well for tree models with high-cardinality features
- Target encoding captures location-specific demand patterns
- Cyclical time encoding preserves temporal continuity
- Ensemble of 3 diverse models reduces variance
- 5-Fold CV ensures robust evaluation


6. TOOLS & LIBRARIES USED
--------------------------
- Python 3.13
- pandas: data loading & manipulation
- numpy: numerical operations
- scikit-learn: KFold CV, R2 metric, LabelEncoder
- LightGBM: gradient boosting model
- XGBoost: gradient boosting model
- CatBoost: gradient boosting model


7. SOURCE FILES
---------------
- dataset/train.csv          : Training data (77,299 rows x 11 columns)
- dataset/test.csv           : Test data (41,778 rows x 10 columns)
- dataset/sample_submission.csv : Submission format
- eda.ipynb                  : Exploratory data analysis notebook
- pipeline.ipynb             : Full training & prediction pipeline
- predicted_demand.csv       : Final predictions output
- approach.txt               : This approach document
================================================================================
"""

with open('./approach.txt', 'w') as f:
    f.write(approach_text)

print('Saved: approach.txt')
print('\nDONE! All files saved successfully.')